## Autoregressive Models

#### 1. Predicting S&P500 returns

In [44]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import yfinance as yf
import plotly.graph_objects as go

##### Simulating AR(1) process

In [2]:
np.random.seed(42)
n = 200
phi = 0.72
c = 0.5 # intercept or constant
errors = np.random.normal(0, 1, n)
y = np.zeros(n)

for t in range(1,n):
    y[t] = c + phi * y[t-1] + errors[t]

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x = np.arange(n),
    y = y,
    mode = 'lines+markers',
))

fig1.update_layout(
    title = 'AR examples_ random generated data',
    xaxis_title = 'timestep',
    yaxis_title = 'price',
    template = 'plotly_white',
    width = 960,
    height = 450
)
fig1.show()

##### Fit AR Model

In [3]:
model = sm.tsa.ARIMA(y, order=(1,0,0)) #AR 1
results = model.fit()
print(results.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                  200
Model:                 ARIMA(1, 0, 0)   Log Likelihood                -269.339
Date:                Sat, 09 Aug 2025   AIC                            544.677
Time:                        20:18:21   BIC                            554.572
Sample:                             0   HQIC                           548.682
                                - 200                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.5898      0.195      8.146      0.000       1.207       1.972
ar.L1          0.6637      0.055     12.090      0.000       0.556       0.771
sigma2         0.8629      0.087      9.919      0.0

##### Applying to S&P500 returns

In [4]:
spy_data = yf.download('^GSPC', start='2019-01-01', end='2025-06-30', auto_adjust=True)
spy_data['returns'] = spy_data['Close'].pct_change()
spy_data.dropna(inplace=True)
spy_data.tail(7)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,returns
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC,
Date,,,,,,
2025-06-18,5980.870117,6018.250000,5971.890137,5987.930176,5106470000,-0.000309
2025-06-20,5967.839844,6018.200195,5952.560059,5999.669922,7451500000,-0.002179
2025-06-23,6025.169922,6028.770020,5943.229980,5969.669922,5597000000,0.009607
2025-06-24,6092.180176,6101.759766,6059.250000,6061.209961,5443690000,0.011122
2025-06-25,6092.160156,6108.509766,6080.089844,6104.229980,5171110000,-0.000003
2025-06-26,6141.020020,6146.520020,6107.270020,6112.089844,5308140000,0.008020
2025-06-27,6173.069824,6187.680176,6132.350098,6150.700195,7889350000,0.005219


In [25]:
# plot returns
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x = spy_data.index,
    y = spy_data['returns'],
    mode = 'lines',
    name= 'SPY_returns'
))

fig2.update_layout(
    title = f'SPY Daily Historical returns plot {spy_data.index[0].strftime('%B_%Y')} and {spy_data.index[-1].strftime('%B_%Y')}',
    xaxis = {'title': 'Dates'},
    yaxis = {'title': 'Returns', 'tickformat': '.2%'},
    template = 'plotly_white',
    width = 960,
    height = 450
)
fig2.show()

###### Fit AR(1) to returns

In [6]:
model_sp = sm.tsa.ARIMA(spy_data['returns'].values, order=(1,0,0))
results_sp = model_sp.fit()

print(results_sp.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 1630
Model:                 ARIMA(1, 0, 0)   Log Likelihood                4801.882
Date:                Sat, 09 Aug 2025   AIC                          -9597.765
Time:                        20:18:22   BIC                          -9581.576
Sample:                             0   HQIC                         -9591.759
                               - 1630                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0006      0.000      2.293      0.022     9.2e-05       0.001
ar.L1         -0.1698      0.010    -16.921      0.000      -0.189      -0.150
sigma2         0.0002   2.26e-06     71.505      0.0

##### Predicting the last 20days

In [7]:
pred_sp20d = results_sp.predict(start=len(spy_data)-20, end=len(spy_data)-1)
pred_sp20d

array([ 5.98385810e-05,  7.54634877e-04,  4.44146394e-05, -2.43910725e-04,
        7.28340902e-04,  1.63683541e-03, -1.00458314e-03,  5.84664404e-04,
       -1.90039964e-04,  1.20670360e-03,  9.18883191e-05,  2.65872904e-03,
       -8.53816289e-04,  2.15886926e-03,  7.93353624e-04,  1.11073986e-03,
       -8.90122352e-04, -1.14737281e-03,  7.41409325e-04, -6.20789326e-04])

In [8]:
## plot prediction vs actual data

actual20 = spy_data['returns'].tail(20).values
last20idx = spy_data.index[-20:]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x = last20idx,
    y = actual20,
    mode = 'lines+markers',
    line=dict(dash='dash', color='blue'),
    marker=dict(size=10, color='blue'),
    name='actual'
))

fig3.add_trace(go.Scatter(
    x = last20idx,
    y = pred_sp20d,
    mode = 'lines+markers',
    line=dict(color='red'),
    marker=dict(size=10, color='red'),
    name='predicted'
))

fig3.update_layout(
    title='Prediction vs Actual (last 20 days)',
    xaxis = {'title': 'Dates'},
    yaxis = {'title': 'Returns', 'tickformat': '.2%'},
    template = 'plotly_white',
    width = 960,
    height = 450
)
fig3.show()

##### Forecast next 10 days

In [9]:
forecast_sp = results_sp.forecast(steps=10)
forecast_sp

array([-0.00014522,  0.00076551,  0.00061089,  0.00063714,  0.00063268,
        0.00063344,  0.00063331,  0.00063333,  0.00063333,  0.00063333])

#### 2. Applying AR models to Bond Yields Prediction

In [66]:
# Download 10-year Treasury yield data yfinance
start_date = "2015-01-01"
end_date = "2025-06-30"
yields_ust10 = yf.download("^TNX", start=start_date, end=end_date, auto_adjust=True)/100 # divide by 100 to convert to percentage
yields_ust10 = yields_ust10['Close']  # Extract closing price
yields_ust10.name = 'Yield'
yields_ust10.tail()

[*********************100%***********************]  1 of 1 completed


Ticker,^TNX
Date,
2025-06-23,0.04320
2025-06-24,0.04293
2025-06-25,0.04293
2025-06-26,0.04253
2025-06-27,0.04283


In [67]:
# plot returns
fig4 = go.Figure()

fig4.add_trace(go.Scatter(
    x = yields_ust10.index,
    y = yields_ust10['^TNX'].values,
    mode = 'lines',
))

fig4.update_layout(
    title = f'10yr_USTreasury Historical yiels between {yields_ust10.index[0].strftime('%B_%Y')} and {yields_ust10.index[-1].strftime('%B_%Y')}',
    xaxis = {'title': 'Dates'},
    yaxis = {'title': 'Returns', 'tickformat': '.2%'},
    template = 'plotly_white',
    width = 960,
    height = 450
)
fig4.show()

###### Removing trends from Bond data

In [68]:
# Checking for non-stationarity in Bond Trends
result_yld = adfuller(yields_ust10.dropna())
print("ADF Statistic:", result_yld[0])
print("p-value:", result_yld[1])

if result_yld[1] > 0.05: # p-values greater than 0.05
    print("Series is non-stationary — differencing needed.")

ADF Statistic: -0.7469141622133757
p-value: 0.8342207292993533
Series is non-stationary — differencing needed.


In [69]:
yields_diff = yields_ust10.diff().dropna()

In [70]:
#### Fitting AR model on difference data

model_yld = sm.tsa.ARIMA(yields_diff.values, order=(1, 0, 0))  # AR(1)
results_yld = model_yld.fit()
print(results_yld.summary())

                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 2635
Model:                 ARIMA(1, 0, 0)   Log Likelihood               16065.398
Date:                Sat, 09 Aug 2025   AIC                         -32124.796
Time:                        20:52:54   BIC                         -32107.166
Sample:                             0   HQIC                        -32118.412
                               - 2635                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       8.197e-06   1.06e-05      0.777      0.437   -1.25e-05    2.89e-05
ar.L1         -0.0060      0.014     -0.441      0.659      -0.033       0.021
sigma2      2.962e-07   5.71e-09     51.828      0.0

/Users/calebfowowe/Documents/Programming/python_wrks/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals



##### Forecasting future yields

In [80]:
# Forecast differences for next 20 days
forecast_diff = results_yld.forecast(steps=20)

# Convert back to yield levels
last_yield = yields_ust10.iloc[-1].values
forecast_yields = last_yield + forecast_diff.cumsum()

print("Forecasted Yields (%):")
print(forecast_yields)

Forecasted Yields (%):
[0.04283645 0.04284465 0.04285285 0.04286105 0.04286925 0.04287744
 0.04288564 0.04289384 0.04290203 0.04291023 0.04291843 0.04292663
 0.04293482 0.04294302 0.04295122 0.04295942 0.04296761 0.04297581
 0.04298401 0.04299221]


In [87]:
## plot prediction vs actual data
last20idx_yld = yields_ust10.index[-20:]

fig5 = go.Figure()
fig5.add_trace(go.Scatter(
    x = last20idx_yld,
    y = forecast_yields,
    mode = 'lines+markers',
    line=dict(dash='solid', color='red'),
    marker=dict(size=6, color='blue'),
    name='actual'
))

fig5.update_layout(
    title='10yr_USTr Yields 20days forecast',
    xaxis = {'title': 'Dates'},
    yaxis = {'title': 'Returns', 'tickformat': '.2%'},
    template = 'plotly_white',
    width = 960,
    height = 450
)
fig5.show()